# Loan feature engineering

`build_features()` converts the raw predictors into numeric features. It selects the allowed predictor columns and excludes the split identifier. No statistics are learned.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from ml_gearbox import audit as ga

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data/raw/data.csv"
assert RAW_PATH.is_file(), f"Missing raw data: {RAW_PATH}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import get_or_create_split
from src.features import build_features
from src.schema import TARGET, ROW_ID

In [2]:
df = pd.read_csv(RAW_PATH)
split = get_or_create_split(df, path=PROJECT_ROOT / "data/processed/split.parquet")
train = split["train"]

engineered = build_features(train.drop(columns=TARGET))
assert TARGET not in engineered and ROW_ID not in engineered
assert engineered.index.equals(train.index)
display(ga.schema_report(engineered))

,dtype,missing_count,missing_pct,n_unique,unique_pct
emp_length_years,float64,28727,6.922769,12,0.002892
revol_util,float64,336,0.080971,1166,0.280988
dti,float64,292,0.070368,6719,1.619177
loan_to_income,float64,280,0.067476,63894,15.397480
payment_to_income,float64,280,0.067476,262202,63.186686
inq_last_6mths,float64,1,0.000241,7,0.001687
loan_amnt,float64,0,0.000000,1543,0.371839
installment,float64,0,0.000000,47935,11.551604
int_rate,float64,0,0.000000,240,0.057836
fico_range_low,float64,0,0.000000,38,0.009157


`term` becomes 36 or 60 months. Employment below one year becomes 0; ten or more years becomes 10. Missing employment stays missing and receives an indicator.

`loan_to_income` divides loan amount by annual income. `payment_to_income` divides annual installments by annual income. Zero income produces missing ratios and an indicator.

In [3]:
display(ga.numeric_profile(engineered))
display(ga.missingness_report(engineered))
display(ga.missingness_report(engineered[["loan_to_income", "payment_to_income"]]))

,count,mean,std,min,25%,50%,75%,max,median,skew,n_unique
loan_to_income,414684.0,0.794809,127.500656,0.000436,0.114286,0.187500,0.281879,40000.00,0.187500,259.936551,63893
payment_to_income,414684.0,0.280236,42.554612,0.000179,0.042965,0.068602,0.103530,13207.92,0.068602,248.416914,262201
annual_inc,414964.0,78906.693146,73774.129071,0.000000,48000.000000,66000.000000,95000.000000,9300000.00,66000.000000,39.795272,26678
annual_inc_zero,414964.0,0.000675,0.025967,0.000000,0.000000,0.000000,0.000000,1.00,0.000000,38.458103,2
dti,414672.0,18.793956,14.812837,-1.000000,11.830000,17.820000,24.570000,999.00,17.820000,31.081790,6718
pub_rec,414964.0,0.244990,0.631790,0.000000,0.000000,0.000000,0.000000,61.00,0.000000,9.419092,29
delinq_2yrs,414964.0,0.328086,0.909476,0.000000,0.000000,0.000000,0.000000,29.00,0.000000,5.693265,26
emp_length_missing,414964.0,0.069228,0.253841,0.000000,0.000000,0.000000,0.000000,1.00,0.000000,3.394046,2
pub_rec_bankruptcies,414964.0,0.154852,0.401105,0.000000,0.000000,0.000000,0.000000,9.00,0.000000,3.079687,10
inq_last_6mths,414963.0,0.587356,0.870168,0.000000,0.000000,0.000000,1.000000,5.00,0.000000,1.726268,6


,feature,missing_count,percentage
0,emp_length_years,28727,6.922769
1,revol_util,336,0.080971
2,dti,292,0.070368
3,loan_to_income,280,0.067476
4,payment_to_income,280,0.067476
5,inq_last_6mths,1,0.000241


,feature,missing_count,percentage
0,loan_to_income,280,0.067476
1,payment_to_income,280,0.067476


In [4]:
display(pd.DataFrame({
    "rows": [len(train), len(engineered)],
    "columns": [train.drop(columns=[TARGET, ROW_ID]).shape[1], engineered.shape[1]],
}, index=["raw_predictors", "engineered_predictors"]))

,rows,columns
raw_predictors,414964,15
engineered_predictors,414964,19
